# Measurement Error in Causal Inference
**Bielicke, L. and Xochihua-Tlecuitl, T.**

---

## Key Ideas

This tutorial covers the following topics related to measurement error in causal inference:

1. Define measurement error and its different types (classical, systematic, differential) and understand how each type affects causal estimates
2. Set up the causal estimand of interest and derive attenuation bias when treatment is mismeasured
3. Explain why measurement error in conditioning variables biases causal estimates
4. Apply instrumental variables / 2SLS to correct for attenuation bias in treatment and confounding variables
5. Understand how SEM, scoring methods, and IRT relate to measurement error
6. Recognize the role of measurement invariance for valid causal estimates

---

## Setup

In [ ]:
# Run this cell first
!pip install -q graphviz ipywidgets

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import statsmodels.api as sm
from ipywidgets import interact, FloatSlider, fixed
import graphviz

np.random.seed(42)
plt.rcParams['figure.dpi'] = 100

---
## 1. Measurement Error: Definitions & Types

> **Measurement Error:** The *misalignment* between the **true state** of a variable and its *reported state*.

In causal inference we typically assume we observe the variables we want to reason about. In practice, we often observe a *proxy* — a noisy or imperfect stand-in for the true variable.

**Why does this matter?**
- The variable we draw conclusions *with* is not always the variable we draw conclusions *about*
- Measurement error in treatment and conditioning variables can bias the causal effect
- Researchers must consider how measured variables differ from target variables

### 1.1 Classical Measurement Error

> **Classical Measurement Error:** The observed variable equals the true value *plus* noise:
> $$X^* = X + \varepsilon$$
> The noise $\varepsilon$ is **independent** of the true variable and all other variables in the model.

**Key Assumptions:**
- $\mathbb{E}(\varepsilon) = 0$ — zero mean
- $\varepsilon \perp X$ — independent of true value
- $\varepsilon \perp Y$ — independent of outcome
- $\varepsilon \perp Z$ — independent of covariates

**Examples:**
- Temperature sensors introduce random noise around the true reading
- Blood pressure measured at a single visit fluctuates around the patient's true level
- Self-reported hours worked vary as participants round up or down
- Happiness measured with culturally biased scales
- Two exam versions where some questions capture more relevant information than others

### 1.2 Systematic Measurement Error

> **Systematic Measurement Error:** The error is consistently biased in one direction — $\mathbb{E}(\varepsilon) \neq 0$.

Unlike classical error, systematic error does not average out with larger samples. Estimates remain biased regardless of sample size.

**Examples:**
- A miscalibrated instrument that always reads above or below the true value
- Survey scales where social desirability leads respondents to consistently over-report

<em>[Add further discussion here]</em>

### 1.3 Differential Measurement Error

Classical error assumes $U_X$ is random and uncorrelated with everything else. When the error is correlated with the outcome or a confounder, it becomes **differential**.

> **Differential Error:** The measurement error is correlated with the outcome or a confounder.

**DAG Implication:** The observed proxy $X^*$ becomes a **collider**. Conditioning on $X^*$ opens a spurious backdoor path.

**Extension:** In psychometrics, this is known as **Differential Item Functioning (DIF)** — when an item's difficulty or discrimination differs across groups independently of the true trait.

<em>[Add further discussion here]</em>

In [ ]:
# DAG: We observe Y*, but Y is latent
g = graphviz.Digraph()
g.attr(rankdir='LR')
g.node('X', shape='square')
g.node('Y', shape='circle', style='dashed')
g.node('Y*', shape='square')
g.node('U_Y*', shape='circle', style='dashed')
g.edge('X', 'Y')
g.edge('Y', 'Y*')
g.edge('U_Y*', 'Y*', style='dashed')
g

### 1.4 Motivating Example: Subjective Well-Being

What is the effect of **happiness** on **physical activity**? Researchers cannot observe happiness directly, so they use self-report scales.

**Measure Reliability:**
> Previous research suggests single-item measures of subjective well-being tend to be correlated at about 0.5 with true happiness, while multi-item scales reach about 0.8. Neither is a perfect measure, but self-reported scores provide meaningful signal.

<em>[Add further discussion here]</em>

In [ ]:
# DAG: Differential ME — U_X correlated with Y, X* is a collider
g = graphviz.Digraph()
g.attr(rankdir='LR')
g.node('X', shape='circle', style='dashed', label='X (latent)')
g.node('X*', shape='square', label='X* (proxy)')
g.node('Y', shape='square')
g.node('U_X', shape='circle', style='dashed')
g.edge('X', 'X*')
g.edge('X', 'Y')
g.edge('U_X', 'X*')
g.edge('U_X', 'Y')  # differential: error correlated with outcome
g

---
## 2. Causal Estimand, Identification & Attenuation Bias

### 2.1 Causal Setup & Identification Assumptions

We want to identify the **Average Treatment Effect (ATE)**:
$$\tau = \mathbb{E}[Y(1) - Y(0)]$$

Standard identification requires:
- **Unconfoundedness:** $Y(x) \perp X \mid Z$ — no unmeasured confounders
- **SUTVA:** No interference between units; stable treatments

Measurement error in $X$, $Z$, or $Y$ can each violate these conditions in different ways, as developed in the subsections below.

<em>[Add further discussion here]</em>

### 2.2 Classical ME in Treatment: Attenuation Bias

We want to estimate the effect of $X$ on $Y$, but we only observe a noisy proxy $X^*$:
$$X^* = X + U_{X^*}$$

The OLS estimator using the proxy is:
$$\hat{\beta}_{biased} = \frac{\text{cov}(Y, X^*)}{\text{var}(X^*)}$$

Substituting $X^* = X + U$ and $Y = \beta X + \varepsilon$:

$$= \frac{\text{cov}(Y, X^*)}{\text{var}(X^*)} = \frac{\text{cov}(\beta X + \varepsilon,\ X + U)}{\text{var}(X) + \text{var}(U)} = \frac{\beta\,\text{var}(X)}{\text{var}(X) + \text{var}(U)} = \beta \cdot \lambda$$

where $\lambda = \dfrac{\text{var}(X)}{\text{var}(X) + \text{var}(U)}$ is the **reliability ratio**, and $0 < \lambda < 1$.

> **Attenuation Bias:** When a regressor is measured with classical error, the coefficient is biased toward zero. The noisier the measure, the greater the bias.

*Example: If $\lambda \approx 0.50$, a true $\beta = 0.4$ would appear as $\hat{\beta}_{biased} \approx 0.2$*

In [ ]:
def plot_attenuation(reliability, n=500, beta_true=0.6):
    """
    Simulate attenuation bias as reliability of X decreases.
    reliability = var(X) / (var(X) + var(U))  i.e. lambda
    """
    var_x = 1.0
    var_u = var_x * (1 - reliability) / reliability  # solve for var(U)

    X_true = np.random.normal(0, np.sqrt(var_x), n)
    U = np.random.normal(0, np.sqrt(var_u), n)
    X_obs = X_true + U
    eps = np.random.normal(0, 1, n)
    Y = beta_true * X_true + eps

    # OLS on observed X
    beta_hat = np.cov(Y, X_obs)[0, 1] / np.var(X_obs)
    beta_theoretical = beta_true * reliability

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(X_obs, Y, alpha=0.3, s=10, color='steelblue')
    xline = np.linspace(X_obs.min(), X_obs.max(), 100)
    ax.plot(xline, beta_true * xline, 'g-', lw=2, label=f'True slope: β={beta_true}')
    ax.plot(xline, beta_hat * xline, 'r--', lw=2, label=f'OLS slope: β̂={beta_hat:.3f}')
    ax.plot(xline, beta_theoretical * xline, 'orange', lw=1.5, ls=':', label=f'Theoretical: βλ={beta_theoretical:.3f}')
    ax.set_xlabel('Observed X*')
    ax.set_ylabel('Y')
    ax.set_title(f'Attenuation Bias  (λ = {reliability:.2f})')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f'  True β: {beta_true}  |  Estimated β̂: {beta_hat:.3f}  |  Theoretical βλ: {beta_theoretical:.3f}')

interact(
    plot_attenuation,
    reliability=FloatSlider(value=1.0, min=0.1, max=1.0, step=0.05, description='Reliability (λ)'),
    n=fixed(500),
    beta_true=fixed(0.6)
);

### 2.3 Classical ME in Outcome Variables

Now suppose the outcome is mismeasured:
$$Y^* = Y + U_Y$$

Substituting the structural equation $Y = \beta X + \varepsilon$:
$$Y^* = \beta X + (\varepsilon + U_Y)$$

The noise $U_Y$ is absorbed into the residual. Since $U_Y \perp X$ under classical ME:

> $\beta$ remains **unbiased**, but residual variance increases — standard errors inflate and $R^2$ decreases.

<em>[Add further discussion here]</em>

In [ ]:
def plot_outcome_me(reliability_y, n=500, beta_true=0.6):
    var_y = 1.0
    var_uy = var_y * (1 - reliability_y) / reliability_y

    X = np.random.normal(0, 1, n)
    eps = np.random.normal(0, 1, n)
    Y_true = beta_true * X + eps
    U_Y = np.random.normal(0, np.sqrt(var_uy), n)
    Y_obs = Y_true + U_Y

    beta_hat = np.cov(Y_obs, X)[0, 1] / np.var(X)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(X, Y_obs, alpha=0.3, s=10, color='steelblue')
    xline = np.linspace(X.min(), X.max(), 100)
    ax.plot(xline, beta_true * xline, 'g-', lw=2, label=f'True slope: β={beta_true}')
    ax.plot(xline, beta_hat * xline, 'r--', lw=2, label=f'OLS slope: β̂={beta_hat:.3f}')
    ax.set_xlabel('X')
    ax.set_ylabel('Observed Y*')
    ax.set_title(f'ME in Outcome  (Reliability = {reliability_y:.2f})')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f'  True β: {beta_true}  |  Estimated β̂: {beta_hat:.3f}  (unbiased, but SE inflated)')

interact(
    plot_outcome_me,
    reliability_y=FloatSlider(value=1.0, min=0.1, max=1.0, step=0.05, description='Reliability (Y)'),
    n=fixed(500),
    beta_true=fixed(0.6)
);

---
## 3. ME in Conditioning Variables

**Goal:** Estimate the causal effect of $X$ on $Y$.

**Problem:** We do not observe $Z$ (the true confounder). We observe a noisy proxy:
$$Z^* = Z + U_{Z^*}$$

Conditioning on $Z^*$ instead of $Z$ **leaves an open backdoor path** from $X$ to $Y$ through $Z$.

> Unlike ME in treatment, this can bias estimates in either direction depending on the structure of the DAG.

<em>[Add further discussion and worked example here]</em>

In [ ]:
# DAG: ME in conditioning variable
g = graphviz.Digraph()
g.attr(rankdir='TB')

with g.subgraph() as s:
    s.attr(rank='source')
    s.node('Z', 'Z (Unobserved)', shape='circle', style='dashed')

g.node('Z*', 'Z* (Proxy)', shape='square')
g.node('U_Z*', shape='circle', style='dashed')
g.node('X', 'X (Treatment)', shape='square')
g.node('Y', 'Y (Outcome)', shape='square')

g.edge('Z', 'X')
g.edge('Z', 'Y')
g.edge('Z', 'Z*', style='dashed', label='P(z*|z)')
g.edge('U_Z*', 'Z*', style='dashed')
g.edge('X', 'Y')
g

---
## 4. Instrumental Variables & Two Stage Least Squares

### 4.1 Setup

To correct for attenuation bias, find a second proxy $T$ for $X$ (an **instrument**).

**Assumptions:**
- $T \perp Y \mid X$ — instrument only affects $Y$ through $X$
- $U_T \perp U_{X^*}$ — instrument errors independent of treatment errors

The IV estimator:
$$\frac{\text{cov}(Y, T)}{\text{cov}(X^*, T)} = \frac{\text{cov}(\beta X + \varepsilon,\ T)}{\text{cov}(X + U,\ T)} = \frac{\beta\,\text{cov}(X, T)}{\text{cov}(X, T)} = \beta$$

### 4.2 Two Stage Least Squares (2SLS)

$$\hat{\beta} = \frac{\text{cov}(Y, Z)}{\text{cov}(X^*, Z)}$$

**Stage 1:** Regress $X^*$ on the instrument $Z$:
$$\hat{X}^* = \hat{\pi}_0 + \hat{\pi}_1 Z$$

**Stage 2:** Regress $Y$ on $\hat{X}^*$:
$$Y = \alpha + \beta \hat{X}^* + \eta$$

> Because $\hat{X}^*$ is formed *entirely* from the exogenous instrument $Z$, it is mathematically guaranteed to be uncorrelated with $\eta$. This d-separation is what allows $\beta$ to be unbiased.

<em>[Add further discussion here]</em>

In [ ]:
def simulate_iv(reliability=0.5, n=1000, beta_true=0.6, seed=42):
    rng = np.random.default_rng(seed)

    var_x = 1.0
    var_u = var_x * (1 - reliability) / reliability

    X_true = rng.normal(0, 1, n)
    U = rng.normal(0, np.sqrt(var_u), n)
    X_obs = X_true + U

    # Instrument: correlated with X_true, independent of U and eps
    Z = X_true + rng.normal(0, 0.5, n)

    eps = rng.normal(0, 1, n)
    Y = beta_true * X_true + eps

    # OLS (biased)
    beta_ols = np.cov(Y, X_obs)[0, 1] / np.var(X_obs)

    # 2SLS
    Z_c = sm.add_constant(Z)
    stage1 = sm.OLS(X_obs, Z_c).fit()
    X_hat = stage1.fittedvalues
    X_hat_c = sm.add_constant(X_hat)
    stage2 = sm.OLS(Y, X_hat_c).fit()
    beta_2sls = stage2.params[1]

    print(f'True β:       {beta_true:.3f}')
    print(f'OLS β̂:        {beta_ols:.3f}  (attenuated by λ={reliability:.2f})')
    print(f'2SLS β̂:       {beta_2sls:.3f}  (corrected)')

interact(
    simulate_iv,
    reliability=FloatSlider(value=0.5, min=0.1, max=0.99, step=0.05, description='Reliability'),
    n=fixed(1000),
    beta_true=fixed(0.6)
);

---
## 5. Latent Variable Methods: SEM, Scoring & IRT

### 5.1 Structural Equation Modeling (SEM)

Adding more covariates to OLS does **not** fix attenuation bias caused by ME in treatment — the variance of the proxy $X^*$ is fundamentally contaminated.

**Two options:**
1. Find an exogenous shock that isolates the unconfounded variance (IV)
2. Model the error explicitly using multiple indicators to partition the variance (**SEM**)

**SEM Measurement Model:**
$$y' = \Lambda f + \varepsilon$$
$$y' = By' + \Gamma x + \zeta$$

- **Purified Variance:** Extracts item-specific noise ($\varepsilon$) away from the structural disturbance
- **Restores Power:** Smaller standard errors for the structural path

> The latent variable $Y$ represents the true outcome, perfectly d-separated from measurement noise $\varepsilon$.

<em>[Add SEM worked example here]</em>

### 5.2 Scoring Methods

How we extract the latent (proxy) variable dictates how measurement error propagates into the causal model.

| Scoring Method | Assumption | Consequence |
|---|---|---|
| **Sum Scores** | Perfect reliability ($\lambda = 1$) | Retains all ME; guarantees attenuation bias |
| **Regression-Based Factor Scores** | Partial reliability | Shrinks toward population mean; biases standard errors |
| **Multiple Imputation** | Draws from posterior | Preserves true variance; allows ME to be accurately reflected via Rubin's Rules |

<em>[Add worked example comparing the three scoring methods here]</em>

### 5.3 Dichotomous Variables & IRT

If the proxy is binary (Yes/No), the classical error conceptualization ($\varepsilon$) does not directly apply. **Item Response Theory (IRT)** links the probability of a \"Yes\" to a true latent state $\theta$.

**The 2PL Model:**
$$P(X^* = 1 \mid \theta) = \frac{1}{1 + e^{-a(\theta - b)}}$$

Two parameters:
- **Threshold ($b$):** How high must the true trait $\theta$ be to have a 50% chance of reporting \"Yes\"?
- **Discrimination ($a$):** How effectively does this item separate the truly treated from the truly untreated?

> Cleaner latent classification than raw scores.

In [ ]:
def plot_icc(discrimination=1.0, threshold=0.0):
    """Plot the Item Characteristic Curve (ICC) for the 2PL IRT model."""
    theta = np.linspace(-4, 4, 300)
    p = 1 / (1 + np.exp(-discrimination * (theta - threshold)))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(theta, p, 'steelblue', lw=2)
    ax.axhline(0.5, color='gray', ls='--', lw=1)
    ax.axvline(threshold, color='red', ls='--', lw=1, label=f'Threshold b={threshold:.1f}')
    ax.annotate(f'Discrimination a={discrimination:.1f}', xy=(threshold + 0.2, 0.5 + 0.05),
                fontsize=10, color='darkgreen')
    ax.set_xlabel('True Trait (θ)')
    ax.set_ylabel('P(X* = 1)')
    ax.set_title('Item Characteristic Curve (2PL IRT)')
    ax.set_ylim(0, 1)
    ax.legend()
    plt.tight_layout()
    plt.show()

interact(
    plot_icc,
    discrimination=FloatSlider(value=1.0, min=0.1, max=4.0, step=0.1, description='Discrimination (a)'),
    threshold=FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.25, description='Threshold (b)')
);

---
## 6. Measurement Invariance

For a causal estimate $\beta$ to be valid, the **measurement model must operate identically** across all levels of the treatment and covariates.

**Types of Invariance:**
- **Configural & Metric Invariance:** The same construct is being measured, and a 1-unit change in the latent trait means the same thing across groups
- **Scalar Invariance:** Groups share the same intercepts — crucial for comparing latent means

> **The Causal Implication of Non-Invariance:**
> If a treatment ($X$) directly alters an item's difficulty or discrimination independently of the true trait ($Y$), we have **DIF**. In the language of DAGs, DIF is a newly opened, unblocked backdoor path that fatally confounds the causal estimate.

<em>[Add worked example here]</em>

---
## 7. Takeaways

- Ignoring measurement error can imply not accounting for a 'hidden' collider — effect estimates will carry that noise
- Proxy variables act as the latent variable only under the assumption that there is validity in that relation
- Before reasoning about confounding and generalizability, researchers often assume variables were measured accurately — this is often unreasonable for abstract constructs or human-reported measures
- Well-designed instruments (e.g., carefully selected IVs) can help detect unbiased effects when researchers suspect measurement error

<em>[Add any additional closing remarks here]</em>

---
## Problem Set Questions

### Question 1

<em>[Insert problem set question 1 here]</em>

---

### Question 2

<em>[Insert problem set question 2 here]</em>